# Patient Mesh Views (Modular)

This notebook provides modular, single-patient visualizations:
1. Raw patient (red)
2. Raw normal (grey)
3. Aligned patient (red)
4. Overlay: normal (grey) + aligned patient colored by distance (blue close → red far)


In [ ]:
# Core imports + plotly inline setup
import os
from pathlib import Path

import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"


In [ ]:
# Paths and CSV loading (data/cropped_data)
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "cropped_data"

clinical_path = DATA_DIR / "hvsmr_clinical.csv"
technical_path = DATA_DIR / "hvsmr_technical.csv"

print("Project root:", PROJECT_ROOT)
print("Clinical path:", clinical_path)
print("Technical path:", technical_path)

if not clinical_path.exists() or not technical_path.exists():
    raise FileNotFoundError("Expected hvsmr CSVs under data/cropped_data")

hvsmr_clinical = pd.read_csv(clinical_path)
hvsmr_technical = pd.read_csv(technical_path)

print("hvsmr_clinical shape:", hvsmr_clinical.shape)
print("hvsmr_technical shape:", hvsmr_technical.shape)


In [ ]:
# Map Pat -> segmentation file (*_cropped_seg.nii.gz)
from pathlib import Path

seg_dir = DATA_DIR
seg_paths = sorted(seg_dir.rglob("*_cropped_seg.nii.gz"))
print("Seg dir:", seg_dir)
print("Found seg files:", len(seg_paths))


def _parse_pat_from_name(p: Path):
    stem = p.name
    suffix = "_cropped_seg.nii.gz"
    if stem.endswith(suffix):
        prefix = stem[: -len(suffix)]
    else:
        prefix = p.stem
    tokens = prefix.replace("-", "_").split("_")
    for t in tokens:
        if t.lower().startswith("pat") and t[3:].isdigit():
            return int(t[3:])
        if t.isdigit():
            return int(t)
    try:
        return int(prefix)
    except Exception:
        return None


pat_to_seg = {}
ambiguous = []
for sp in seg_paths:
    pat = _parse_pat_from_name(sp)
    if pat is None:
        ambiguous.append(sp)
        continue
    if pat in pat_to_seg and pat_to_seg[pat] != sp:
        ambiguous.append(sp)
    else:
        pat_to_seg[pat] = sp

print("Mapped Pats:", len(pat_to_seg))
if ambiguous:
    print("Ambiguous files (showing up to 10):")
    for p in ambiguous[:10]:
        print(" -", p)


In [ ]:
# Load a segmentation as a mesh (marching cubes)
try:
    import nibabel as nib
except ModuleNotFoundError:
    !pip -q install nibabel
    import nibabel as nib

try:
    from skimage.measure import marching_cubes
except ModuleNotFoundError:
    !pip -q install scikit-image
    from skimage.measure import marching_cubes


def extract_meshes_from_seg(seg_nii_path, labels="all_nonzero"):
    img = nib.load(str(seg_nii_path))
    seg = img.get_fdata()
    zooms = img.header.get_zooms()[:3]

    u = np.unique(seg)
    nonzero = u[u != 0]
    if labels == "all_nonzero":
        use_labels = nonzero
    else:
        use_labels = np.array(labels, dtype=float)

    out = {}
    for lab in use_labels:
        mask = (seg == float(lab)).astype(np.uint8)
        if mask.sum() < 10:
            continue
        verts, faces, normals, values = marching_cubes(mask, level=0.5, spacing=zooms)
        out[float(lab)] = (verts.astype(np.float32), faces.astype(np.int32))
    return out


In [ ]:
# Load normal OBJ mesh
obj_path = PROJECT_ROOT / "SubTool-0-7412864.OBJ"
if not obj_path.exists():
    raise FileNotFoundError("Could not locate SubTool-0-7412864.OBJ")


def load_obj_vertices_faces(path: Path):
    verts = []
    faces = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("v "):
                parts = line.split()
                if len(parts) >= 4:
                    verts.append([float(parts[1]), float(parts[2]), float(parts[3])])
            elif line.startswith("f "):
                parts = line.split()[1:]
                idx = []
                for p in parts:
                    v_str = p.split("/")[0]
                    if v_str:
                        idx.append(int(v_str) - 1)
                if len(idx) >= 3:
                    for k in range(1, len(idx) - 1):
                        faces.append([idx[0], idx[k], idx[k + 1]])
    verts = np.asarray(verts, dtype=np.float32)
    faces = np.asarray(faces, dtype=np.int32)
    return verts, faces


normal_verts, normal_faces = load_obj_vertices_faces(obj_path)
print("Normal mesh verts:", normal_verts.shape, "faces:", normal_faces.shape)


In [ ]:
# Build per-patient meshes + point clouds for ICP
from sklearn.neighbors import NearestNeighbors

rng = np.random.default_rng(0)

NORMAL_SAMPLE_N = 60000
PATIENT_SAMPLE_N = 20000
DEFAULT_LABEL = 3.0

normal_n = normal_verts.shape[0]
if normal_n <= NORMAL_SAMPLE_N:
    normal_pts = normal_verts.astype(np.float32)
else:
    idx = rng.choice(normal_n, size=NORMAL_SAMPLE_N, replace=False)
    normal_pts = normal_verts[idx].astype(np.float32)

pat_meshes = {}
for pat, sp in sorted(pat_to_seg.items()):
    meshes = extract_meshes_from_seg(sp, labels="all_nonzero")
    pat_meshes[pat] = meshes


def get_patient_pts_for_icp(pat: int, default_label: float = DEFAULT_LABEL, n_sample: int = PATIENT_SAMPLE_N):
    meshes = pat_meshes[pat]
    if default_label in meshes:
        v = meshes[default_label][0]
        used_label = default_label
    else:
        labs = sorted(meshes.keys())
        used_label = labs[0]
        v = meshes[used_label][0]
    if v.shape[0] <= n_sample:
        pts = v.astype(np.float32)
    else:
        idx = rng.choice(v.shape[0], size=n_sample, replace=False)
        pts = v[idx].astype(np.float32)
    return pts, used_label


In [ ]:
# Rigid ICP (Kabsch)
import numpy as np


def rigid_transform_kabsch(A: np.ndarray, B: np.ndarray):
    assert A.shape == B.shape and A.shape[1] == 3
    centroid_A = A.mean(axis=0)
    centroid_B = B.mean(axis=0)
    AA = A - centroid_A
    BB = B - centroid_B
    H = AA.T @ BB
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T
    t = centroid_B - centroid_A @ R.T
    return R.astype(np.float32), t.astype(np.float32)


def icp_rigid(source_pts: np.ndarray, target_pts: np.ndarray, max_iter: int = 50, tol: float = 1e-5):
    src = source_pts.astype(np.float32).copy()
    tgt = target_pts.astype(np.float32)
    nn = NearestNeighbors(n_neighbors=1, algorithm="auto").fit(tgt)
    R_total = np.eye(3, dtype=np.float32)
    t_total = np.zeros(3, dtype=np.float32)
    prev_rmse = np.inf
    history = []
    for it in range(max_iter):
        dists, idx = nn.kneighbors(src, return_distance=True)
        matched = tgt[idx[:, 0]]
        rmse = float(np.sqrt((dists[:, 0] ** 2).mean()))
        history.append(rmse)
        R, t = rigid_transform_kabsch(src, matched)
        src = src @ R.T + t
        R_total = R @ R_total
        t_total = t + (t_total @ R.T)
        if abs(prev_rmse - rmse) < tol:
            break
        prev_rmse = rmse
    return R_total, t_total, history


In [ ]:
# Visualization helpers

def _center(xyz: np.ndarray):
    xyz = xyz.astype(np.float32)
    c = xyz.mean(axis=0)
    return xyz - c, c


def _dist_to_rgb(dist: np.ndarray, vmin: float = 0.0, vmax: float = None):
    if vmax is None:
        vmax = float(np.quantile(dist, 0.95))
    x = np.clip((dist - vmin) / (vmax - vmin + 1e-12), 0.0, 1.0)
    r = (255.0 * x).astype(np.uint8)
    g = (255.0 * (1.0 - np.abs(x - 0.5) * 2.0)).astype(np.uint8)
    b = (255.0 * (1.0 - x)).astype(np.uint8)
    colors = [f"rgb({ri},{gi},{bi})" for ri, gi, bi in zip(r, g, b)]
    return colors, float(vmax)


In [ ]:
# Modular view functions
GREY = "rgb(170,170,170)"
RED = "rgb(220,40,40)"


def render_raw_patient(pat: int, center_only_for_view: bool = True, point_size: float = 2.0):
    pts, used_label = get_patient_pts_for_icp(int(pat))
    if center_only_for_view:
        pts, _ = _center(pts)
    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
                mode="markers",
                name=f"Patient {pat} (raw)",
                marker=dict(size=point_size, color=RED, opacity=0.95),
            )
        ]
    )
    fig.update_layout(
        title=f"Raw patient {pat} | label={used_label}",
        scene=dict(aspectmode="data"),
        margin=dict(l=0, r=0, b=0, t=60),
        height=720,
        paper_bgcolor="rgb(0,0,0)",
    )
    fig.show()


def render_raw_normal(center_only_for_view: bool = True, point_size: float = 2.0):
    pts = normal_pts.astype(np.float32)
    if center_only_for_view:
        pts, _ = _center(pts)
    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
                mode="markers",
                name="Normal (raw)",
                marker=dict(size=point_size, color=GREY, opacity=0.85),
            )
        ]
    )
    fig.update_layout(
        title=f"Raw normal heart (n={pts.shape[0]})",
        scene=dict(aspectmode="data"),
        margin=dict(l=0, r=0, b=0, t=60),
        height=720,
        paper_bgcolor="rgb(0,0,0)",
    )
    fig.show()


def render_aligned_patient(pat: int, center_only_for_view: bool = True, point_size: float = 2.0):
    src_pts, used_label = get_patient_pts_for_icp(int(pat))
    R, t, hist = icp_rigid(src_pts.astype(np.float32), normal_pts.astype(np.float32), max_iter=60, tol=1e-6)
    aligned = src_pts.astype(np.float32) @ R.T + t
    if center_only_for_view:
        aligned, _ = _center(aligned)
    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=aligned[:, 0], y=aligned[:, 1], z=aligned[:, 2],
                mode="markers",
                name=f"Patient {pat} (aligned)",
                marker=dict(size=point_size, color=RED, opacity=0.95),
            )
        ]
    )
    fig.update_layout(
        title=f"Aligned patient {pat} | label={used_label} | ICP iters={len(hist)} rmse_end={hist[-1]:.4f}",
        scene=dict(aspectmode="data"),
        margin=dict(l=0, r=0, b=0, t=60),
        height=720,
        paper_bgcolor="rgb(0,0,0)",
    )
    fig.show()


def render_overlay(pat: int, center_only_for_view: bool = True, normal_point_size: float = 2.0, patient_point_size: float = 2.0):
    src_pts, used_label = get_patient_pts_for_icp(int(pat))
    R, t, hist = icp_rigid(src_pts.astype(np.float32), normal_pts.astype(np.float32), max_iter=60, tol=1e-6)
    aligned = src_pts.astype(np.float32) @ R.T + t

    nn_norm = NearestNeighbors(n_neighbors=1).fit(normal_pts.astype(np.float32))
    d_p2n, _ = nn_norm.kneighbors(aligned, return_distance=True)
    d_p2n = d_p2n[:, 0].astype(np.float32)
    colors, vmax_used = _dist_to_rgb(d_p2n, vmin=0.0, vmax=None)

    normal_view = normal_pts.astype(np.float32)
    patient_view = aligned
    if center_only_for_view:
        normal_view, _ = _center(normal_view)
        patient_view, _ = _center(patient_view)

    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=normal_view[:, 0], y=normal_view[:, 1], z=normal_view[:, 2],
                mode="markers",
                name="Normal (grey)",
                marker=dict(size=normal_point_size, color=GREY, opacity=0.80),
            ),
            go.Scatter3d(
                x=patient_view[:, 0], y=patient_view[:, 1], z=patient_view[:, 2],
                mode="markers",
                name=f"Patient {pat} (colored)",
                marker=dict(size=patient_point_size, color=colors, opacity=0.95),
            ),
        ]
    )
    fig.update_layout(
        title=(
            f"Overlay: normal (grey) + patient {pat} (colored) | label={used_label} | "
            f"ICP iters={len(hist)} rmse_end={hist[-1]:.4f} | vmax={vmax_used:.3f}"
        ),
        scene=dict(aspectmode="data"),
        margin=dict(l=0, r=0, b=0, t=70),
        height=760,
        paper_bgcolor="rgb(0,0,0)",
    )
    fig.show()


In [ ]:
# Run for a single patient (default: 1)
PAT_IDS = [1]  # set to list of Pats, e.g., [1, 5, 10]
# To run all mapped patients, uncomment the next line:
# PAT_IDS = sorted(pat_to_seg.keys())

for pat in PAT_IDS:
    render_raw_patient(pat)
    render_raw_normal()
    render_aligned_patient(pat)
    render_overlay(pat)
